# STEP 08a - 속도/거주/유동인구
**08a 완료 후 → COMPAS_08b 실행**

In [ ]:
import os

BASE_DIR   = os.getcwd()          # /opt/app-root/src
EPDO_DIR   = os.getcwd()
DATA_DIR   = os.path.join(BASE_DIR, "data")
OUTPUT_DIR = os.path.join(EPDO_DIR, "epdo_analysis", "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"BASE_DIR : {BASE_DIR}")
print(f"DATA_DIR : {DATA_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")


In [ ]:
import csv
import json
import os
import warnings
from collections import defaultdict, Counter

import geopandas as gpd
from shapely.geometry import Point

warnings.filterwarnings("ignore", category=UserWarning)   # CRS 관련 경고 억제


# 입력
GAP_FILE     = os.path.join(OUTPUT_DIR, "06_인프라공백_고위험격자.csv")
STEP07_FILE  = os.path.join(OUTPUT_DIR, "07_링크_속도혼잡_보강.csv")
GRID_FILE    = os.path.join(BASE_DIR, "data", "01._격자_(4개_시·구).geojson")
RES_FILE     = os.path.join(BASE_DIR, "data", "03._성연령별_거주인구(격자).csv")
FLOAT_FILE   = os.path.join(BASE_DIR, "data", "04._성연령별_유동인구.csv")
WORK_FILE    = os.path.join(BASE_DIR, "data", "05._시간대별_직장인구.csv")
VISIT_FILE   = os.path.join(BASE_DIR, "data", "06._시간대별_방문인구.csv")
SVC_FILE     = os.path.join(BASE_DIR, "data", "07._주중주말_서비스인구.csv")
ROAD_FILE    = os.path.join(BASE_DIR, "data", "08.상세도로망_네트워크.geojson")
LU_FILE      = os.path.join(BASE_DIR, "data", "22._토지이용계획도_(4개_신도시).geojson")
LU_HANAM     = os.path.join(BASE_DIR, "data", "23._토지이용계획도_(하남교산).geojson")

# 출력
OUTPUT_PATH  = os.path.join(OUTPUT_DIR, "08_격자_종합위험지수.csv")
HANAM_OUTPUT = os.path.join(OUTPUT_DIR, "08_하남교산_예측위험도.csv")

CRS_PROJ     = "EPSG:5186"   # 한국 중부원점 (단위: 미터, centroid 계산 정확)
CRS_GEO      = "EPSG:4326"   # WGS84

# 교통약자 취약 시간대: 등교(07~09), 하교(13~16), 퇴근(17~19)
VULN_SLOTS   = {7, 8, 9, 13, 14, 15, 16, 17, 18, 19}
SLOT_COLS    = [f"TMST_{str(h).zfill(2)}" for h in VULN_SLOTS]
ALL_SLOTS    = [f"TMST_{str(h).zfill(2)}" for h in range(24)]

# 토지이용 → 위험 카테고리 매핑
SCHOOL_TYPES    = {"학교", "초등학교", "중학교", "고등학교", "교육시설"}
RESIDENT_TYPES  = {"아파트", "연립주택", "다세대주택", "단독주택", "공동주택", "주상복합", "연립"}
COMMERCIAL_TYPES = {"상업", "일반상업", "근린상업", "근린생활시설용지", "상업시설"}


# ── 유틸 ────────────────────────────────────────────────────────────────────

def load_csv_dict(path, key_col):
    with open(path, "r", encoding="utf-8-sig") as f:
        return {row[key_col]: row for row in csv.DictReader(f)}


def safe_float(val, default=0.0):
    try:
        return float(val or 0)
    except (ValueError, TypeError):
        return default


def points_to_grid(lon_col, lat_col, rows, value_cols, grid_gdf):
    """lon/lat 포인트 → 격자 공간 join 후 격자별 집계 반환"""
    pts = []
    for row in rows:
        try:
            lon, lat = float(row[lon_col]), float(row[lat_col])
        except (ValueError, KeyError):
            continue
        rec = {"geometry": Point(lon, lat)}
        for c in value_cols:
            rec[c] = safe_float(row.get(c))
        pts.append(rec)

    pts_gdf  = gpd.GeoDataFrame(pts, crs=CRS_GEO)
    joined   = gpd.sjoin(pts_gdf, grid_gdf[["gid", "geometry"]], how="left", predicate="within")
    return joined


def median(values):
    sv = sorted(v for v in values if v is not None)
    if not sv:
        return 0
    mid = len(sv) // 2
    return (sv[mid - 1] + sv[mid]) / 2 if len(sv) % 2 == 0 else sv[mid]


# ── 메인 ─────────────────────────────────────────────────────────────────────


In [ ]:
print("=" * 60)
print("STEP 08 - 격자 종합 위험 지수 산출 (v2)")
print("=" * 60)


## ── 1. 고위험 격자 목록

In [ ]:
print("\n[1] 고위험 격자 로드 중...")
gap_data = load_csv_dict(GAP_FILE, "grid_gid")
print(f"    고위험 격자 수: {len(gap_data):,}개")


## ── 2. 격자 GeoDataFrame (고위험만 필터)

In [ ]:
print("\n[2] 격자 폴리곤 로드 중...")
grid_gdf = gpd.read_file(GRID_FILE)
grid_gdf = grid_gdf[grid_gdf["gid"].isin(gap_data.keys())].reset_index(drop=True)
print(f"    고위험 격자 폴리곤: {len(grid_gdf):,}개")


## ── 3. STEP 07 속도·혼잡 → 격자 단위 집계

In [ ]:
# 개선 v3:
#   [문제1] 중심점 기반 → 링크 선분 intersects 기반으로 변경
#           (중심점이 다른 격자에 떨어지는 오류 해소)
#   [문제2] 속도 미계측 링크 → 도로 등급별 평균으로 보간
print("\n[3] STEP 07 속도·혼잡 → 격자 단위 집계 중...")
step07 = load_csv_dict(STEP07_FILE, "link_id")

# 도로망 로드 (speed 데이터 있는 링크만)
road_gdf_all = gpd.read_file(ROAD_FILE)
road_gdf_all["link_id"] = road_gdf_all["link_id"].astype(str)

road_gdf_spd = road_gdf_all[road_gdf_all["link_id"].isin(step07.keys())].copy()

# 속도·혼잡 컬럼 추가
road_gdf_spd["avg_speed"]       = road_gdf_spd["link_id"].map(
    lambda lid: safe_float(step07.get(lid, {}).get("avg_speed")))
road_gdf_spd["congestion_risk"] = road_gdf_spd["link_id"].map(
    lambda lid: safe_float(step07.get(lid, {}).get("congestion_risk")))

# [문제1 해결] centroid 제거 → 선분 geometry 그대로 intersects 조인
road_proj = road_gdf_spd.to_crs(CRS_PROJ)
grid_proj = grid_gdf.to_crs(CRS_PROJ)

link_grid = gpd.sjoin(
    road_proj[["link_id", "avg_speed", "congestion_risk", "geometry"]],
    grid_proj[["gid", "geometry"]],
    how="inner",
    predicate="intersects"   # 선분이 격자와 교차하면 모두 매칭
)
speed_agg = {}
for gid, grp in link_grid.groupby("gid"):
    spd_vals = [v for v in grp["avg_speed"] if v > 0]
    cng_vals = [v for v in grp["congestion_risk"] if v > 0]
    speed_agg[str(gid)] = {
        "grid_avg_speed":       round(sum(spd_vals) / len(spd_vals), 2) if spd_vals else None,
        "grid_congestion_risk": round(sum(cng_vals) / len(cng_vals), 4) if cng_vals else None,
    }
matched_before_interp = len(speed_agg)
print(f"    [문제1 해결 후] 속도·혼잡 매칭 격자: {matched_before_interp:,}개 "
      f"({matched_before_interp/len(gap_data)*100:.1f}%)")

# [문제2 해결] 속도 미계측 링크 → 도로 등급별 평균 보간
# 1단계: 전체 도로망에서 road_rank별 평균 속도 계산 (계측 링크 기준)
rank_speed_map = defaultdict(list)
rank_cong_map  = defaultdict(list)
for lid, data in step07.items():
    spd = safe_float(data.get("avg_speed"))
    cng = safe_float(data.get("congestion_risk"))
    # 해당 link_id의 road_rank 조회
    rank_rows = road_gdf_all[road_gdf_all["link_id"] == lid]
    if rank_rows.empty:
        continue
    rank = str(rank_rows.iloc[0].get("road_rank", ""))
    if spd > 0 and rank:
        rank_speed_map[rank].append(spd)
    if cng > 0 and rank:
        rank_cong_map[rank].append(cng)

rank_avg_speed = {r: round(sum(v)/len(v), 2) for r, v in rank_speed_map.items()}
rank_avg_cong  = {r: round(sum(v)/len(v), 4) for r, v in rank_cong_map.items()}
overall_avg_speed = round(sum(rank_avg_speed.values())/len(rank_avg_speed), 2) \
                    if rank_avg_speed else 30.0
overall_avg_cong  = round(sum(rank_avg_cong.values())/len(rank_avg_cong), 4) \
                    if rank_avg_cong else 0.0

# 2단계: 결측 격자에 해당 격자 내 링크의 도로 등급별 평균 적용
# 전체 도로망(속도 없는 링크 포함)과 격자 intersects 조인
road_all_proj = road_gdf_all.to_crs(CRS_PROJ)
link_grid_all = gpd.sjoin(
    road_all_proj[["link_id", "road_rank", "geometry"]],
    grid_proj[["gid", "geometry"]],
    how="inner",
    predicate="intersects"
)
interp_cnt = 0
for gid, grp in link_grid_all.groupby("gid"):
    gid_str = str(gid)
    if gid_str in speed_agg:
        continue   # 이미 속도 데이터 있음
    # 격자 내 링크들의 도로 등급 목록
    ranks = [str(r) for r in grp["road_rank"].dropna() if str(r)]
    if not ranks:
        continue
    spd_vals = [rank_avg_speed.get(r, overall_avg_speed) for r in ranks]
    cng_vals = [rank_avg_cong.get(r,  overall_avg_cong)  for r in ranks]
    speed_agg[gid_str] = {
        "grid_avg_speed":       round(sum(spd_vals)/len(spd_vals), 2),
        "grid_congestion_risk": round(sum(cng_vals)/len(cng_vals), 4),
        "interpolated": True,   # 보간 여부 표시
    }
    interp_cnt += 1

total_matched = len(speed_agg)
print(f"    [문제2 해결 후] 도로등급 보간 추가: {interp_cnt:,}개")
print(f"    최종 속도·혼잡 커버 격자: {total_matched:,}개 "
      f"({total_matched/len(gap_data)*100:.1f}%)")
print(f"    도로 등급별 기본 속도: { {k: v for k, v in sorted(rank_avg_speed.items())} }")


## ── 4. 03번 거주인구 — gid 직접 join

In [ ]:
print("\n[4] 거주인구(노인 60대+) 로드 중...")
res_data = {}
ELDERLY_COLS = ["m_60g_pop", "w_60g_pop", "m_70g_pop", "w_70g_pop",
                "m_80g_pop", "w_80g_pop", "m_90g_pop", "w_90g_pop",
                "m_100g_pop", "w_100g_pop"]
ALL_RES_COLS = ["m_20g_pop", "w_20g_pop", "m_30g_pop", "w_30g_pop",
                "m_40g_pop", "w_40g_pop", "m_50g_pop", "w_50g_pop"] + ELDERLY_COLS
with open(RES_FILE, "r", encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        gid = row.get("gid", "")
        if gid not in gap_data:
            continue
        elderly = sum(safe_float(row.get(c)) for c in ELDERLY_COLS)
        total   = sum(safe_float(row.get(c)) for c in ALL_RES_COLS)
        res_data[gid] = {"elderly_pop": elderly, "total_res_pop": total}
print(f"    거주인구 매칭 격자: {len(res_data):,}개 "
      f"(비어있는 격자는 거주인구 0으로 처리)")


## ── 5. 04번 유동인구 — 공간 join

In [ ]:
print("\n[5] 유동인구(어린이·노인) 공간 join 중...")
float_cols = ["m_10g_pop", "w_10g_pop", "m_60g_pop", "w_60g_pop",
              "m_20g_pop", "w_20g_pop", "m_30g_pop", "w_30g_pop",
              "m_40g_pop", "w_40g_pop", "m_50g_pop", "w_50g_pop"]
with open(FLOAT_FILE, "r", encoding="utf-8-sig") as f:
    float_rows = list(csv.DictReader(f))
float_joined = points_to_grid("lon", "lat", float_rows, float_cols, grid_gdf)

float_agg = {}
for gid, grp in float_joined.dropna(subset=["gid"]).groupby("gid"):
    child   = grp["m_10g_pop"].sum() + grp["w_10g_pop"].sum()
    elderly = grp["m_60g_pop"].sum() + grp["w_60g_pop"].sum()
    total   = sum(grp[c].sum() for c in float_cols)
    float_agg[str(gid)] = {
        "child_float":   round(child, 4),
        "elderly_float": round(elderly, 4),
        "total_float":   round(total, 4),
    }
print(f"    유동인구 매칭 격자: {len(float_agg):,}개")


In [ ]:
import json as _j, os as _os
_tmp = os.path.join(OUTPUT_DIR, "_tmp_step08")
_os.makedirs(_tmp, exist_ok=True)
for _name, _obj in [
    ("speed_agg",  speed_agg),
    ("res_data",   res_data),
    ("float_agg",  float_agg),
]:
    with open(_os.path.join(_tmp, f"{_name}.json"), "w", encoding="utf-8") as _f:
        _j.dump(_obj, _f, ensure_ascii=False)
print("저장 완료 → COMPAS_08b 실행")
